In [1]:
import numpy as np
import pandas as pd

In [2]:
train = pd.read_csv("/Users/nidhishgupta/Desktop/GridlockChallenge/data/raw/train.csv")
test = pd.read_csv("/Users/nidhishgupta/Desktop/GridlockChallenge/data/raw/test.csv")

In [3]:
print(train.head())
print(train.info())
print(train.isnull().sum())
print(train.nunique())

   Index geohash  day timestamp    demand     RoadType  NumberofLanes  \
0      0  qp02z1   48       0:0  0.048804          NaN              1   
1      1  qp02zt   48       0:0  0.118507  Residential              3   
2      2  qp08bj   48       0:0  0.027132  Residential              1   
3      3  qp08gt   48       0:0  0.003272  Residential              1   
4      4  qp02zq   48       0:0  0.010819  Residential              1   

  LargeVehicles Landmarks  Temperature Weather  
0   Not Allowed        No          NaN     NaN  
1       Allowed       Yes    31.104565   Sunny  
2   Not Allowed        No    25.919267   Sunny  
3   Not Allowed        No          NaN   Rainy  
4   Not Allowed        No    10.803667   Rainy  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 

In [5]:
train['timestamp'].unique()

array(['0:0', '0:15', '0:30', '0:45', '1:0', '1:15', '1:30', '1:45',
       '2:0', '2:15', '2:30', '2:45', '3:0', '3:15', '3:30', '3:45',
       '4:0', '4:15', '4:30', '4:45', '5:0', '5:15', '5:30', '5:45',
       '6:0', '6:15', '6:30', '6:45', '7:0', '7:15', '7:30', '7:45',
       '8:0', '8:15', '8:30', '8:45', '9:0', '9:15', '9:30', '9:45',
       '10:0', '10:15', '10:30', '10:45', '11:0', '11:15', '11:30',
       '11:45', '12:0', '12:15', '12:30', '12:45', '13:0', '13:15',
       '13:30', '13:45', '14:0', '14:15', '14:30', '14:45', '15:0',
       '15:15', '15:30', '15:45', '16:0', '16:15', '16:30', '16:45',
       '17:0', '17:15', '17:30', '17:45', '18:0', '18:15', '18:30',
       '18:45', '19:0', '19:15', '19:30', '19:45', '20:0', '20:15',
       '20:30', '20:45', '21:0', '21:15', '21:30', '21:45', '22:0',
       '22:15', '22:30', '22:45', '23:0', '23:15', '23:30', '23:45'],
      dtype=object)

In [9]:
# =========================================================
# TRAFFIC DEMAND PREDICTION
# HIGH R2 ENSEMBLE MODEL
# =========================================================

# pip install catboost lightgbm xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

# =========================================================
# LOAD DATA
# =========================================================

train = pd.read_csv("/Users/nidhishgupta/Desktop/GridlockChallenge/data/raw/train.csv")
test = pd.read_csv("/Users/nidhishgupta/Desktop/GridlockChallenge/data/raw/test.csv")

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)

# =========================================================
# REMOVE DUPLICATES
# =========================================================

train = train.drop_duplicates()

# =========================================================
# HANDLE NULL VALUES
# =========================================================

cat_cols = [
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather",
    "geohash"
]

num_cols = [
    "NumberofLanes",
    "Temperature"
]

# fill categorical nulls
for col in cat_cols:

    train[col] = train[col].fillna("Missing")
    test[col] = test[col].fillna("Missing")

# fill numerical nulls
for col in num_cols:

    median_value = train[col].median()

    train[col] = train[col].fillna(median_value)
    test[col] = test[col].fillna(median_value)

# =========================================================
# TIMESTAMP PROCESSING
# timestamp format:
# '0:0', '0:15', '1:30', ...
# =========================================================

def process_timestamp(df):

    df["timestamp"] = df["timestamp"].astype(str)

    split_time = df["timestamp"].str.split(":", expand=True)

    df["hour"] = split_time[0].astype(int)
    df["minute"] = split_time[1].astype(int)

    # total minutes from midnight
    df["time_minutes"] = (
        df["hour"] * 60 + df["minute"]
    )

    # quarter-hour slot
    df["quarter"] = df["minute"] // 15

    # traffic peak features
    df["is_morning_peak"] = (
        ((df["hour"] >= 7) & (df["hour"] <= 10))
    ).astype(int)

    df["is_evening_peak"] = (
        ((df["hour"] >= 16) & (df["hour"] <= 20))
    ).astype(int)

    df["is_night"] = (
        ((df["hour"] >= 22) | (df["hour"] <= 5))
    ).astype(int)

    # cyclical encoding
    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24
    )

    df["minute_sin"] = np.sin(
        2 * np.pi * df["minute"] / 60
    )

    df["minute_cos"] = np.cos(
        2 * np.pi * df["minute"] / 60
    )

    return df

train = process_timestamp(train)
test = process_timestamp(test)

# =========================================================
# DAY FEATURES
# =========================================================

day_mapping = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}

if train["day"].dtype == "object":

    train["day_num"] = train["day"].map(day_mapping)
    test["day_num"] = test["day"].map(day_mapping)

    # fallback encoding
    if train["day_num"].isnull().sum() > 0:

        le_day = LabelEncoder()

        combined_days = pd.concat([
            train["day"].astype(str),
            test["day"].astype(str)
        ])

        le_day.fit(combined_days)

        train["day_num"] = le_day.transform(
            train["day"].astype(str)
        )

        test["day_num"] = le_day.transform(
            test["day"].astype(str)
        )

else:

    train["day_num"] = train["day"]
    test["day_num"] = test["day"]

# weekend feature
train["is_weekend"] = (
    train["day_num"] >= 5
).astype(int)

test["is_weekend"] = (
    test["day_num"] >= 5
).astype(int)

# =========================================================
# LIGHT GEOHASH FEATURES
# =========================================================

train["geo_len"] = (
    train["geohash"].astype(str).apply(len)
)

test["geo_len"] = (
    test["geohash"].astype(str).apply(len)
)

train["geo_4"] = (
    train["geohash"].astype(str).str[:4]
)

test["geo_4"] = (
    test["geohash"].astype(str).str[:4]
)

train["geo_5"] = (
    train["geohash"].astype(str).str[:5]
)

test["geo_5"] = (
    test["geohash"].astype(str).str[:5]
)

# =========================================================
# K-FOLD
# =========================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# =========================================================
# SAFE TARGET ENCODING
# =========================================================

target_encode_cols = [
    "geohash",
    "geo_4",
    "geo_5",
    "RoadType",
    "Weather"
]

global_mean = train["demand"].mean()

for col in target_encode_cols:

    train[f"{col}_te"] = np.nan

    for tr_idx, val_idx in kf.split(train):

        tr_fold = train.iloc[tr_idx]
        val_fold = train.iloc[val_idx]

        means = (
            tr_fold.groupby(col)["demand"].mean()
        )

        train.loc[
            val_fold.index,
            f"{col}_te"
        ] = val_fold[col].map(means)

    train[f"{col}_te"] = (
        train[f"{col}_te"].fillna(global_mean)
    )

    full_means = (
        train.groupby(col)["demand"].mean()
    )

    test[f"{col}_te"] = (
        test[col].map(full_means)
    )

    test[f"{col}_te"] = (
        test[f"{col}_te"].fillna(global_mean)
    )

# =========================================================
# LABEL ENCODING
# =========================================================

all_cat_cols = [
    "geohash",
    "geo_4",
    "geo_5",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather",
    "day"
]

for col in all_cat_cols:

    le = LabelEncoder()

    combined = pd.concat([
        train[col].astype(str),
        test[col].astype(str)
    ])

    le.fit(combined)

    train[col] = le.transform(
        train[col].astype(str)
    )

    test[col] = le.transform(
        test[col].astype(str)
    )

# =========================================================
# FEATURES
# IMPORTANT:
# DROP RAW TIMESTAMP COLUMN
# =========================================================

drop_cols = [
    "Index",
    "timestamp"
]

X = train.drop(
    columns=drop_cols + ["demand"]
)

y = train["demand"]

X_test = test.drop(columns=drop_cols)

print("\nTotal Features:", X.shape[1])

# =========================================================
# TRAINING
# =========================================================

oof_pred = np.zeros(len(train))
test_pred = np.zeros(len(test))

fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(
    kf.split(X, y)
):

    print(f"\n========== FOLD {fold+1} ==========")

    X_train = X.iloc[tr_idx]
    y_train = y.iloc[tr_idx]

    X_valid = X.iloc[val_idx]
    y_valid = y.iloc[val_idx]

    # =====================================================
    # CATBOOST
    # =====================================================

    cat_model = CatBoostRegressor(
        iterations=3500,
        learning_rate=0.025,
        depth=8,
        loss_function='RMSE',
        eval_metric='R2',
        random_seed=42,
        subsample=0.8,
        reg_lambda=3,
        min_data_in_leaf=20,
        verbose=0
    )

    cat_model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    # =====================================================
    # LIGHTGBM
    # =====================================================

    lgb_model = LGBMRegressor(
        n_estimators=4000,
        learning_rate=0.02,
        max_depth=10,
        num_leaves=128,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.5,
        reg_lambda=1.0,
        min_child_samples=20,
        random_state=42
    )

    lgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="l2"
    )

    # =====================================================
    # XGBOOST
    # =====================================================

    xgb_model = XGBRegressor(
        n_estimators=3500,
        learning_rate=0.02,
        max_depth=9,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.3,
        reg_lambda=1.5,
        min_child_weight=3,
        objective='reg:squarederror',
        tree_method='hist',
        random_state=42
    )

    xgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    # =====================================================
    # VALIDATION PREDICTIONS
    # =====================================================

    cat_val = cat_model.predict(X_valid)
    lgb_val = lgb_model.predict(X_valid)
    xgb_val = xgb_model.predict(X_valid)

    # weighted ensemble
    val_pred = (
        0.50 * cat_val +
        0.30 * lgb_val +
        0.20 * xgb_val
    )

    score = r2_score(y_valid, val_pred)

    print("Fold R2 Score:", score)

    fold_scores.append(score)

    oof_pred[val_idx] = val_pred

    # =====================================================
    # TEST PREDICTIONS
    # =====================================================

    cat_test = cat_model.predict(X_test)
    lgb_test = lgb_model.predict(X_test)
    xgb_test = xgb_model.predict(X_test)

    fold_test = (
        0.50 * cat_test +
        0.30 * lgb_test +
        0.20 * xgb_test
    )

    test_pred += fold_test / 5

# =========================================================
# FINAL SCORE
# =========================================================

final_r2 = r2_score(y, oof_pred)

print("\n===================================")
print("FINAL CV R2:", final_r2)
print("COMPETITION SCORE:", final_r2 * 100)
print("===================================")

# =========================================================
# FEATURE IMPORTANCE
# =========================================================

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": cat_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

print("\nTop Important Features:")
print(importance_df.head(20))

# =========================================================
# CREATE SUBMISSION
# =========================================================

submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": test_pred
})

submission.to_csv(
    "submission.csv",
    index=False
)

print("\nsubmission.csv created successfully!")
print(submission.head())

Train Shape: (77299, 11)
Test Shape : (41778, 10)

Total Features: 29

========== FOLD 1 ==========
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003015 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1322
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 27
[LightGBM] [Info] Start training from score 0.093784
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 